# Component 1 — NVTV Video Metadata Pipeline (Google Colab)

MSc Video-Retrieval Project · Queen's University Belfast · ECS8056

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: GPU` (a T4 / 16 GB is enough).

Run the cells top to bottom. You will **upload your `.mp4` videos directly** — no dataset path is hardcoded.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the pipeline code
Either clone your repo, or upload the `src/`, `config.yaml`, `requirements.txt` files manually. Edit the URL below.

In [ ]:
import os
REPO_URL = "https://github.com/sauravv4/leveraging-foundational-models-for-video-archival-data.git"
REPO_DIR = "/content/pipeline"
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!ls

## 3. Install dependencies
ffmpeg is pre-installed on Colab.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
!ffmpeg -version | head -n 1

## 4. Upload your videos
Run this cell and pick your `.mp4` files. They go into `/content/videos`.
(You can also drag files into that folder via the Files pane on the left.)

In [ ]:
import os, shutil
from google.colab import files
VIDEO_DIR = "/content/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)
uploaded = files.upload()  # select one or more .mp4 (and optional .txt synopses)
for name in uploaded:
    shutil.move(name, os.path.join(VIDEO_DIR, name))
print("Files now in", VIDEO_DIR, ":")
print(os.listdir(VIDEO_DIR))

## 5. Write a Colab config
Points `dataset_dir` at the upload folder; everything else inherits the repo defaults.

In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["paths"]["dataset_dir"] = "/content/videos"
cfg["paths"]["output_dir"] = "/content/output"
cfg["paths"]["clips_dir"] = "/content/output/clips"
cfg["paths"]["metadata_dir"] = "/content/output/metadata"
cfg["paths"]["embeddings_dir"] = "/content/output/embeddings"
cfg["paths"]["manifest_path"] = "/content/output/manifest.json"
cfg["paths"]["failures_path"] = "/content/output/failures.json"
cfg["paths"]["combined_csv"] = "/content/output/metadata_combined.csv"
# cfg["models"]["whisper_size"] = "large-v3"  # uncomment for best transcripts
with open("colab_config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("Wrote colab_config.yaml")

## 6. Run the pipeline
Stages run in memory-safe order (CLIP+BLIP-2 freed before Whisper loads). Re-running resumes from checkpoints.

In [ ]:
!python run_pipeline.py --config colab_config.yaml

## 7. Inspect the results

In [ ]:
import pandas as pd
df = pd.read_csv("/content/output/metadata_combined.csv")
print(df.shape)
df.head()

In [ ]:
import json
from glob import glob
sample = sorted(glob('/content/output/metadata/*.json'))[0]
print(json.dumps(json.load(open(sample)), indent=2)[:2000])

## 8. (Optional) Download everything

In [ ]:
from google.colab import files
!cd /content && zip -r -q output.zip output
files.download('/content/output.zip')